In [1]:
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.tri import Triangulation

In [2]:
grid_data = {}

# Grid info for deterministic runs:
grid_data["grid_det"] = xr.open_dataset("./grids/icon_grid_0026_R03B07_G.nc")
#grid_data["tri_det"] = Triangulation(np.rad2deg(grid_data["grid_det"]["clon"]), np.rad2deg(grid_data["grid_det"]["clat"]))
grid_data["area_det"] = grid_data["grid_det"]["cell_area"]

# Grid info for ensemble runs:
grid_data["grid_ens"] = xr.open_dataset("./grids/icon_grid_0028_R02B07_N02.nc")
#grid_data["tri_ens"] = Triangulation(np.rad2deg(grid_data["grid_ens"]["clon"]), np.rad2deg(grid_data["grid_ens"]["clat"]))
grid_data["area_ens"] = grid_data["grid_ens"]["cell_area"]


# Define "study area" by cropping deterministic global to ensemble nest
lon_min = np.min(np.rad2deg(grid_data["grid_ens"]["clon"]))
lon_max = np.max(np.rad2deg(grid_data["grid_ens"]["clon"]))

lat_min = np.min(np.rad2deg(grid_data["grid_ens"]["clat"]))
lat_max = np.max(np.rad2deg(grid_data["grid_ens"]["clat"]))

# Crop domain by setting the cell areas to 0
grid_data["cells_det"] = ((np.rad2deg(grid_data["grid_det"]["clon"]) >= lon_min) & (np.rad2deg(grid_data["grid_det"]["clon"]) <= lon_max) & 
                          (np.rad2deg(grid_data["grid_det"]["clat"]) >= lat_min) & (np.rad2deg(grid_data["grid_det"]["clat"]) <= lat_max)).values
grid_data["area_det"][dict(cell=grid_data["grid_det"]["cell"][~grid_data[f"cells_det"]])] = 0

In [3]:
def load_ensemble(datetime, run_names, exp_name, variable):
    # Deterministic:
    ds_det = xr.open_dataset(f"./data/{exp_name}/{datetime}/{variable}_det.nc")[variable].squeeze()

    ds_det_weighted = ds_det.weighted(grid_data["area_det"].rename({"cell": "ncells"}))
    ts_det = ds_det_weighted.mean(dim="ncells", skipna=True)
    
    if variable == "TOT_PREC": #de-accumulate
        ts_det = ts_det.diff(dim="time")
   
    dataframe = pd.Series(ts_det.data, ts_det["time"], name=f"{exp_name}_det").to_frame()
    
    # Ensemble:
    for mem in run_names[:]: #[:] creates a copy, which is looped over
        # Catch missing data due to crashed ensemble member:
        try:
            ds_mem = xr.open_dataset(f"./data/{exp_name}/{datetime}/{variable}_{mem}.nc")[variable].squeeze()
        except:
            run_names.remove(mem)
            continue
        
        ds_mem_weighted = ds_mem.weighted(grid_data["area_ens"].rename({"cell": "ncells"}))
        ts_mem = ds_mem_weighted.mean(dim="ncells", skipna=True)
        
        if variable == "TOT_PREC": #de-accumulate
            ts_mem = ts_mem.diff(dim="time")

        dataframe[f"{exp_name}_{mem}"] = pd.Series(ts_mem.data, ts_mem["time"])

    return dataframe

In [9]:
ini_dates = [2021070912, 2021071012, 2021071112]#, 2021071212]
datetime = ini_dates[2]

exp_names = ["CTRL", "SATU", "WILT", "CO2x2"]
variables = ["CAPE_ML", "CIN_ML", "T_2M", "EVAPT", "TOT_PREC"]

In [18]:
for variable in variables:
    dframes = []
    run_names = [f"mem{i:03d}" for i in range(1,21)]
        
    for exp_name in exp_names:
        dframes.append(load_ensemble(datetime, run_names, exp_name, variable))
    
    with open(f"july21_{variable.lower()}.csv", "w") as file:
        if variable == "T_2M":
            file.write("# Area weighted domain average of two meter temperature given in K\n")
        elif variable == "TOT_PREC":
            file.write("# Area weighted domain average of de-accumulated total precipitation given in kg per m2\n")
        elif variable == "EVAPT":
            file.write("# Area weighted domain average of evapotranspiration given in 1 per kg2 and s\n")

        pd.concat(dframes, axis=1, join="outer").to_csv(file)

If you save some of the concatenated dataframes as variable dframe, this would be the way to plot:

In [ ]:
variable = "CAPE_ML"
dframe = pd.read_csv(f"july21_{variable.lower()}.csv", comment="#", index_col=0, parse_dates=True)

fig, ax = plt.subplots()
fig.tight_layout()


for exp_name, color in zip(exp_names, ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    im = plt.plot(dframe[f"{exp_name}_det"], color=color, label=exp_name)
    for name in [name for name in dframe.columns if f"{exp_name}_mem" in name]:
        plt.plot(dframe[name], color=color, alpha=0.5)

plt.legend()
ax.set(ylabel=r'CAPE / J kg$^{-1}$')

ax.tick_params(axis="x", labelrotation=30)

plt.show()

#plt.plot(dframe.index, sinusoidal_function(dframe.index.hour,*params))

In [31]:
from scipy.optimize import curve_fit

def sinusoidal_function(t, A, B, C, D, E):
    """
    Sinusoidal function to represent diurnal cycle.
    
    Parameters:
    - t: Time (in hours)
    - A: Amplitude
    - B: Phase shift
    - C: Vertical offset
    """
    return A * np.sin(2 * np.pi * t / 24 + B) + C + D * np.sin(2 * np.pi * t / 12 + E)

In [39]:
def remove_diurnal_cycle(dframe, exp_name, initial_guess, sub_mean=False):
    """
    Remove the diurnal cycle from the data in `dframe` corresponding to experiment with name `exp_name`.
    """
    # Get fitting data:
    data = []
    cnames = []

    for cname in dframe.columns:
        
        if not exp_name in cname:
            continue

        cnames.append(cname)
        nan_mask = ~dframe.loc[:,cname].isna()
        data.append(dframe.loc[:,cname][nan_mask])
    
    data = pd.concat(data)

    # Fit:
    params, params_covariance = curve_fit(sinusoidal_function, data.index.hour, data.values, p0=initial_guess)
    if sub_mean:
        fitted_diurnal_cycle = sinusoidal_function(np.array(dframe.index.hour), *params)
    else:
        fitted_diurnal_cycle = sinusoidal_function(np.array(dframe.index.hour), *[params[i] if i != 2 else 0 for i in range(len(params))])

    # Remove cycle:
    df = dframe.loc[:,cnames].subtract(fitted_diurnal_cycle, axis=0)

    return params, df.loc[:,cnames]

In [50]:
first_guesses = {"T_2M": [4,0,295,1,0], "EVAPT": [1e-5,0,-1e-5,1e-6,0], "CAPE_ML": [50,0,120,10,0], "CIN_ML": [150,0,-150,10,0]}

In [ ]:
variable = "T_2M"
dframe = pd.read_csv(f"july21_{variable.lower()}.csv", comment="#", index_col=0, parse_dates=True)

fig, axs = plt.subplots(1, 2, gridspec_kw={"width_ratios": [3,1]}, sharey=True)
fig.tight_layout()

for exp_name, color in zip(exp_names, ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    params, df = remove_diurnal_cycle(dframe, exp_name, initial_guess=first_guesses[variable])

    axs[0].plot(df[f"{exp_name}_det"], color=color, label=exp_name)
    #for cname in df.columns: #spaghetti
    #    plt.plot(df[cname], color=color, alpha=0.2)
    axs[0].fill_between(df.index,
                    df.iloc[:,1:].quantile(q=0.25, axis="columns"), 
                    df.iloc[:,1:].quantile(q=0.75, axis="columns"), 
                    color=color, alpha=0.4)
    
    axs[1].plot(sinusoidal_function(np.arange(24), *params))

if variable == "T_2M":
    axs[0].set(title='2m temperature w/o diurnal cycle', ylabel=r'T / K')
elif variable == "EVAPT":
    axs[0].set(title='Evapotranspiration w/o diurnal cycle', ylabel=r'e / kg$^{-2}$ s$^{-1}$')
elif variable == "CAPE_ML":
    axs[0].set(title='CAPE w/o diurnal cycle', ylabel=r'CAPE / J kg$^{-1}$')
elif variable == "CIN_ML":
    axs[0].set(title='CIN w/o diurnal cycle', ylabel=r'CIN / J kg$^{-1}$')
    
axs[0].tick_params(axis="x", labelrotation=30)

axs[1].set(title="Diurnal cycle", xlabel="Hour of day")

axs[0].legend()
plt.show()

# PRUDENCE Regions



    ‘BI’: Latitude: <50.0 or >59.0, Longitude: <-10.0 or >2.0

    ‘IP’: Latitude: <36.0 or >44.0, Longitude: <-10.0 or >3.0

    ‘FR’: Latitude: <44.0 or >50.0, Longitude: <-5.0 or >5.0

    ‘ME’: Latitude: <48.0 or >55.0, Longitude: <2.0 or >16.0

    ‘SC’: Latitude: <55.0 or >70.0, Longitude: <5.0 or >30.0

    ‘AL’: Latitude: <44.0 or >48.0, Longitude: <5.0 or >15.0

    ‘MD’: Latitude: <36.0 or >44.0, Longitude: <3.0 or >25.0

    ‘EA’: Latitude: <44.0 or >55.0, Longitude: <16.0 or >30.0
